# AD vs FTD: confirming lagged cross-correlation (C3) on ds006036

Experiment 9 (`ADvsFTD_riemann.ipynb`) tried five stage-2 candidates. Its pre-registered pipeline failed
(AUC 0.576), but one candidate, **C3 lagged cross-correlation**, reached 0.628. C3 was picked after the results
were seen, so that number is optimistic. This notebook gives C3 one test on recordings it was not picked on.

- **ds006036** holds the same 88 people, recorded right after ds004504 with eyes open under photic stimulation.
  It is a new recording, **not a new cohort**.
- **One pipeline, nothing selected.** C3 exactly as in Experiment 9: same bands, filter, lags, reference,
  classifier, C grid, folds and seeds. Cell 5 proves the pipeline is unchanged by reproducing Experiment 9's
  C3 scores to the last digit.
- **Less data.** ds006036 recordings are short, and the shortest patient has 11 clean 4 s epochs. Everyone
  therefore contributes 44 s instead of 300 s. A control runs C3 on ds004504 with the same 44 s, to separate
  "less data" from "another recording condition".
- **Design fixed in Cell 2** (`results/c3_ds006036/protocol.json`) before any matrix is computed.
- **Reads** `../data_ds006036/derivatives/`, `../data/derivatives/`, `features/subjects.csv`,
  `features/riemann_matrices.npz` and `results/riemann/stage2_oof.npz`.
- **Writes** `features/c3_ds006036_matrices.npz` and `results/c3_ds006036/`.
- About 1 min, plus ~6 min for the permutation test in Cell 7 (CPU).

In [1]:
# ===========================================================================
# CELL 1 — IMPORTS AND CONFIGURATION
# ===========================================================================
import json, os, time, warnings
from pathlib import Path

import mne
import numpy as np
import pandas as pd
from joblib import Parallel, delayed
from scipy import stats
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")
mne.set_log_level("ERROR")

EXP_DIR = Path.cwd()                          # experiments_ad_vs_ftd/
PROJECT = EXP_DIR.parent
DATA_06 = PROJECT / "data_ds006036"           # eyes open, photic stimulation: the same 88 people
DATA_04 = PROJECT / "data"                    # ds004504, eyes closed: the 44 s control and the identity check
FEAT_DIR = EXP_DIR / "features"
OUT_DIR = EXP_DIR / "results" / "c3_ds006036"
for _root in (DATA_06, DATA_04):
    assert (_root / "participants.tsv").exists(), f"no dataset at {_root}: open this notebook from experiments_ad_vs_ftd/"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ---- data per subject: the most that every one of the 59 patients has --------
SFREQ    = 500.0
EPOCH_S  = 4.0               # Experiment 9's epoch length
N_EPOCHS = 11                # 11 x 4 s = 44 s; the shortest ds006036 patients have exactly 11 join-free epochs
EPOCH_N  = int(EPOCH_S * SFREQ)
CHANNELS = ["Fp1", "Fp2", "F3", "F4", "C3", "C4", "P3", "P4", "O1", "O2",
            "F7", "F8", "T3", "T4", "T5", "T6", "Fz", "Cz", "Pz"]          # subject_cache order
BANDS = {"delta": (0.5, 4.0), "theta": (4.0, 8.0), "alpha": (8.0, 13.0), "beta": (13.0, 30.0)}
HC, AD, FTD = 0, 1, 2

# ---- C3, unchanged from Experiment 9 ------------------------------------------
DECIM   = 4              # cross-correlation on 125 Hz copies of each band
MAX_LAG = 12             # +-96 ms at 125 Hz

# ---- protocol --------------------------------------------------------------
SEED, N_FOLDS, N_REPEATS = 42, 5, 5   # Experiment 9's folds and seeds
INNER_FOLDS = 5
C_GRID      = [0.001, 0.01, 0.1, 1.0]
N_BOOT      = 2000
N_PERM      = 1000                    # label shuffles through the whole pipeline, all 5 repeats (Cell 7)
N_JOBS      = max(1, min(8, (os.cpu_count() or 2) - 2))
FORCE_MATRICES = False                # True = recompute features/c3_ds006036_matrices.npz

print(f"ds006036: {DATA_06}\nds004504: {DATA_04}\noutput  : {OUT_DIR}\nworkers : {N_JOBS}")


ds006036: C:\Projects\Major Project\data_ds006036
ds004504: C:\Projects\Major Project\data
output  : C:\Projects\Major Project\experiments_ad_vs_ftd\results\c3_ds006036
workers : 8


In [2]:
# ===========================================================================
# CELL 2 — PROTOCOL, WRITTEN BEFORE ANY MATRIX IS COMPUTED
# ===========================================================================
PROTOCOL = {
    "question": "Does C3 (lagged cross-correlation), the best of Experiment 9's five candidates and picked after "
                "its results were seen, separate AD (36) from FTD (23) on ds006036, a second recording of the same "
                "patients (eyes open, photic stimulation)? One pipeline, nothing selected",
    "data": "ds006036 derivatives (ASR + ICA, A1-A2 reference). Per subject, the first 11 4 s epochs that contain no "
            "ASR join, packed into the join-free stretches from the start of the recording: 44 s, the most every "
            "one of the 59 patients has. Photic stimulation is left in, as recorded. Common average reference",
    "pipeline": "C3 of Experiment 9 unchanged: per band (delta 0.5-4, theta 4-8, alpha 8-13, beta 13-30 Hz, "
                "zero-phase FIR on the continuous signal), normalised cross-correlation per epoch at 125 Hz, averaged "
                "over epochs at each lag up to +-96 ms, the value at the lag with the largest |r| per pair, Fisher z; "
                "4 x 171 values -> standardise -> L2 logistic regression, balanced class weights, C in "
                "[0.001, 0.01, 0.1, 1] by 5-fold inner CV AUC on the outer training patients",
    "identity_checks": "(1) this notebook's matrix code rebuilds Experiment 9's ds004504 sub-001 cross-correlation "
                       "(300 s) exactly; (2) its cross-validation code reproduces Experiment 9's C3 scores exactly",
    "folds": "Experiment 9's subject-to-fold assignment (the 3-class notebook's folds), 5 repeats, the same seeds; "
             "read from results/riemann/stage2_oof.npz",
    "primary": "AUC of q_AD, AD vs FTD on the 59 patients, mean over the 5 repeats. C3 is confirmed if the one-sided "
               "permutation p < 0.05, from 1000 shuffles of the patients' diagnoses through the whole pipeline "
               "(all 5 repeats)",
    "secondary": {
        "control": "the same pipeline on ds004504 with the same 44 s (same epoch rule): separates 'less data' from "
                   "'another recording condition'",
        "severity": "likelihood-ratio test of each score added to an unpenalised logistic model of MMSE + age + sex; "
                    "Spearman r with MMSE",
        "agreement": "Spearman r between each patient's out-of-fold C3 score on ds006036 and on ds004504 "
                     "(Experiment 9, 300 s)",
    },
    "metrics": "AUC = P(AD scores higher); balanced accuracy at 0.5, recall AD and FTD; pooled per repeat, mean over "
               "repeats; 95 % CI from 2000 stratified subject bootstraps of the repeat mean",
    "limits": "same patients, not an independent cohort; 44 s instead of 300 s; eyes open with flicker instead of "
              "eyes-closed rest. A pass supports C3 across two recordings of these 59 patients only. A fail with the "
              "control passing points to the recording condition; both failing cannot tell less data from no signal",
    "decision": "reported whatever it is. C3 becomes the stage-2 model of the two-stage plan only if the primary "
                "test passes",
}

_path = OUT_DIR / "protocol.json"
if _path.exists():                    # a re-run keeps the original date and must not change the design
    _old = json.loads(_path.read_text(encoding="utf-8"))
    assert {k: v for k, v in _old.items() if k != "written"} == PROTOCOL, "protocol changed after it was written"
    PROTOCOL = _old
else:
    PROTOCOL = {"written": time.strftime("%Y-%m-%d %H:%M") + ", before any matrix was computed", **PROTOCOL}
    _path.write_text(json.dumps(PROTOCOL, indent=1, ensure_ascii=False), encoding="utf-8")
print(f"protocol: {_path}  (written {PROTOCOL['written']})")


protocol: C:\Projects\Major Project\experiments_ad_vs_ftd\results\c3_ds006036\protocol.json  (written 2026-11-16 17:29, before any matrix was computed)


In [3]:
# ===========================================================================
# CELL 3 — SUBJECTS AND EXPERIMENT 9'S FOLDS
# ===========================================================================
subjects = pd.read_csv(FEAT_DIR / "subjects.csv")
assert list(subjects.participant_id) == [f"sub-{i:03d}" for i in range(1, 89)]
LABELS   = subjects.label.to_numpy()
DEMENTIA = np.flatnonzero(LABELS != HC)

# ds006036's participant table must describe the same people
_p06 = pd.read_csv(DATA_06 / "participants.tsv", sep="\t")
assert list(_p06.participant_id) == list(subjects.participant_id)
assert (_p06.Group.map({"A": "AD", "C": "HC", "F": "FTD"}) == subjects.group).all(), "groups differ"
assert (_p06.Gender == subjects.sex).all() and (_p06.Age == subjects.age).all() and (_p06.MMSE == subjects.mmse).all()

_s2 = np.load(EXP_DIR / "results" / "riemann" / "stage2_oof.npz", allow_pickle=True)
assert list(_s2["subjects"]) == list(subjects.participant_id) and (_s2["labels"] == LABELS).all()
FOLD_OF = _s2["fold_of"]                                                     # (5, 88)
FOLDS = [dict(repeat=r + 1, fold=f, test=np.flatnonzero(FOLD_OF[r] == f))
         for r in range(N_REPEATS) for f in range(1, N_FOLDS + 1)]
C3_EXP9 = _s2["q_ad"][list(_s2["candidates"]).index("C3 lagged cross-correlation")]   # (5, 88), 300 s
print(f"{len(subjects)} subjects: " + ", ".join(f"{g} {n}" for g, n in subjects.group.value_counts().items())
      + f"; ds006036 participants match; {len(FOLDS)} folds from Experiment 9")


88 subjects: AD 36, HC 29, FTD 23; ds006036 participants match; 25 folds from Experiment 9


In [4]:
# ===========================================================================
# CELL 4 — CROSS-CORRELATION MATRICES (identity check first)
# ===========================================================================
IU = np.triu_indices(len(CHANNELS), 1)                       # the 171 channel pairs


def set_file(root, sid):
    if root == DATA_06:
        path = root / "derivatives" / "eeglab" / sid / "eeg" / f"{sid}_task-photomark_eeg.set"
    else:
        path = root / "derivatives" / sid / "eeg" / f"{sid}_task-eyesclosed_eeg.set"
    if not path.exists():
        raise FileNotFoundError(path)
    return path


def joins_of(raw):
    return np.sort(np.array([a["onset"] for a in raw.annotations if a["description"] == "boundary"]) * SFREQ)


def grid_starts(raw):
    """Experiment 9's rule: 4 s epochs on a fixed grid from t = 0, dropping any with an ASR join inside."""
    joins = joins_of(raw)
    starts = np.arange(0, raw.n_times - EPOCH_N + 1, EPOCH_N)
    inside = (joins[None, :] > starts[:, None]) & (joins[None, :] < starts[:, None] + EPOCH_N)
    return starts[~inside.any(axis=1)]


def packed_starts(raw):
    """4 s epochs packed into each join-free stretch from its start, in time order."""
    edges = np.concatenate([[0.0], joins_of(raw), [raw.n_times]])
    return np.array([s for a, b in zip(edges[:-1], edges[1:])
                     for s in range(int(np.ceil(a)), int(b) - EPOCH_N + 1, EPOCH_N)], dtype=int)


def lagged_xcorr(ep, max_lag=MAX_LAG):
    """Experiment 9's function, unchanged. ep (epochs, channels, samples) -> per pair the value at the lag with
    the largest |r| (sign kept), and that lag."""
    ep = ep - ep.mean(-1, keepdims=True)
    n = ep.shape[-1]
    norm = np.sqrt((ep ** 2).sum(-1))
    R = np.stack([(np.einsum("eit,ejt->eij", ep[:, :, l:], ep[:, :, :n - l])
                   / (norm[:, :, None] * norm[:, None, :])).mean(0) for l in range(max_lag + 1)])
    full = np.concatenate([R[:0:-1].transpose(0, 2, 1), R])
    k = np.abs(full).argmax(0)
    return np.take_along_axis(full, k[None], 0)[0], k - max_lag


def subject_xcorr(path, starts_rule, n_epochs):
    with warnings.catch_warnings():                       # parallel workers do not inherit Cell 1's settings
        warnings.filterwarnings("ignore")
        raw = mne.io.read_raw_eeglab(path, preload=False, verbose="ERROR")
    starts = starts_rule(raw)[:n_epochs]
    assert len(starts) == n_epochs, f"{path.name}: only {len(starts)} clean epochs"
    x = raw.get_data(picks=CHANNELS, start=0, stop=int(starts[-1]) + EPOCH_N) * 1e6      # µV, A1-A2 reference
    ev = np.linalg.eigvalsh(np.cov(x))
    rank = int((ev > ev.max() * 1e-10).sum())
    x = x - x.mean(0, keepdims=True)                      # common average reference
    xcz, lags = [], []
    for lo, hi in BANDS.values():
        xb = mne.filter.filter_data(x, SFREQ, lo, hi, verbose="ERROR")
        ep = np.stack([xb[:, s:s + EPOCH_N] for s in starts])
        r, lag = lagged_xcorr(ep[:, :, ::DECIM])
        xcz.append(np.arctanh(np.clip(r[IU], -0.999, 0.999)))
        lags.append(lag[IU])
    return np.concatenate(xcz), np.stack(lags), rank, len(starts_rule(raw))


# ---- identity check: Experiment 9's ds004504 sub-001 (75 grid epochs, 300 s) ----
with np.load(FEAT_DIR / "riemann_matrices.npz") as z:
    XCZ_EXP9 = z["xcz"]
_x = subject_xcorr(set_file(DATA_04, "sub-001"), grid_starts, 75)[0]
assert np.allclose(_x, XCZ_EXP9[0], atol=1e-10), "the matrix code differs from Experiment 9"
print("identity check passed: Experiment 9's sub-001 cross-correlation rebuilt exactly")

_mat = FEAT_DIR / "c3_ds006036_matrices.npz"
if _mat.exists() and not FORCE_MATRICES:
    with np.load(_mat) as z:
        XCZ_06, LAGS_06, RANK_06, AVAIL_06, XCZ_04S = z["xcz_06"], z["lags_06"], z["rank_06"], z["avail_06"], z["xcz_04_44s"]
    print(f"loaded {_mat.name}")
else:
    t0 = time.time()
    _r06 = Parallel(n_jobs=N_JOBS)(delayed(subject_xcorr)(set_file(DATA_06, s), packed_starts, N_EPOCHS)
                                   for s in subjects.participant_id)
    _r04 = Parallel(n_jobs=N_JOBS)(delayed(subject_xcorr)(set_file(DATA_04, s), packed_starts, N_EPOCHS)
                                   for s in subjects.participant_id)
    XCZ_06, LAGS_06, RANK_06, AVAIL_06 = (np.stack([r[k] for r in _r06]) for k in range(4))
    XCZ_04S = np.stack([r[0] for r in _r04])
    np.savez_compressed(_mat, xcz_06=XCZ_06, lags_06=LAGS_06, rank_06=RANK_06, avail_06=AVAIL_06,
                        xcz_04_44s=XCZ_04S, subjects=subjects.participant_id.to_numpy(), bands=np.array(list(BANDS)))
    print(f"computed 2 x {len(subjects)} subjects in {time.time() - t0:.0f} s -> {_mat.name}")

assert np.isfinite(XCZ_06).all() and np.isfinite(XCZ_04S).all()
print(f"ds006036 cross-correlation {XCZ_06.shape}; clean 4 s epochs available per group (min / median), "
      f"of which {N_EPOCHS} are used:")
print(pd.DataFrame(dict(group=subjects.group, avail=AVAIL_06)).groupby("group").avail.agg(["min", "median"]).to_string())
print("ICA rank per group:")
print(pd.crosstab(subjects.group, RANK_06, colnames=["rank"]))
print(f"lag of the kept |r| (ms), median |lag| per band: "
      + ", ".join(f"{b} {np.median(np.abs(LAGS_06[:, k])) * DECIM / SFREQ * 1000:.0f}" for k, b in enumerate(BANDS)))


identity check passed: Experiment 9's sub-001 cross-correlation rebuilt exactly


computed 2 x 88 subjects in 9 s -> c3_ds006036_matrices.npz
ds006036 cross-correlation (88, 684); clean 4 s epochs available per group (min / median), of which 11 are used:
       min  median
group             
AD      11    65.0
FTD     13    49.0
HC      38    87.0
ICA rank per group:
rank   13  14  15  16  17  18  19
group                            
AD      1   1   3   4  18   8   1
FTD     0   0   2   2  12   6   1
HC      0   0   0   6  15   6   2
lag of the kept |r| (ms), median |lag| per band: delta 8, theta 0, alpha 0, beta 0


In [5]:
# ===========================================================================
# CELL 5 — THE C3 PIPELINE (identity check first) AND THE CROSS-VALIDATION
# ===========================================================================
def run_fold(X, test, labels, seed):
    """Experiment 9's C3: tuned on the dementia patients outside `test`, q_AD for every test patient."""
    train = np.setdiff1d(np.flatnonzero(labels != HC), test)
    inner = StratifiedKFold(INNER_FOLDS, shuffle=True, random_state=seed)
    pipe = make_pipeline(StandardScaler(), LogisticRegression(C=0.1, class_weight="balanced", max_iter=5000))
    est = GridSearchCV(pipe, {"logisticregression__C": C_GRID}, cv=inner, scoring="roc_auc")
    return est.fit(X[train], (labels[train] == AD).astype(int)).predict_proba(X[test])[:, 1]


def run_cv(X, labels):
    q = np.full((N_REPEATS, len(labels)), np.nan)
    for s in FOLDS:
        q[s["repeat"] - 1, s["test"]] = run_fold(X, s["test"], labels, SEED + 100 * s["repeat"] + s["fold"])
    return q


t0 = time.time()
assert np.allclose(run_cv(XCZ_EXP9, LABELS), C3_EXP9, atol=1e-10), "the pipeline differs from Experiment 9's C3"
print("identity check passed: Experiment 9's C3 scores reproduced exactly")
Q = {"ds006036, 44 s (primary)": run_cv(XCZ_06, LABELS),
     "ds004504, same 44 s (control)": run_cv(XCZ_04S, LABELS),
     "ds004504, 300 s (Experiment 9, post hoc)": C3_EXP9}
print(f"cross-validation done in {time.time() - t0:.0f} s")


identity check passed: Experiment 9's C3 scores reproduced exactly


cross-validation done in 10 s


In [6]:
# ===========================================================================
# CELL 6 — AD VS FTD, WITH CIs
# ===========================================================================
YD = (LABELS[DEMENTIA] == AD).astype(int)
POS, NEG = np.flatnonzero(YD == 1), np.flatnonzero(YD == 0)


def auc(pos_scores, neg_scores):
    """Mann-Whitney AUC = P(an AD score > an FTD score), ties count one half. Works on stacked rows."""
    d = pos_scores[..., :, None] - neg_scores[..., None, :]
    return (d > 0).mean((-1, -2)) + 0.5 * (d == 0).mean((-1, -2))


def per_repeat(q):
    qd = q[:, DEMENTIA]
    call = qd >= 0.5
    rec_ad, rec_ftd = call[:, POS].mean(1), (~call[:, NEG]).mean(1)
    return pd.DataFrame(dict(auc=auc(qd[:, POS], qd[:, NEG]), bal_acc=(rec_ad + rec_ftd) / 2,
                             recall_AD=rec_ad, recall_FTD=rec_ftd))


_rng = np.random.default_rng(SEED)
_bp = _rng.choice(POS, (N_BOOT, len(POS)))
_bn = _rng.choice(NEG, (N_BOOT, len(NEG)))
rows = []
for n, q in Q.items():
    pr = per_repeat(q)
    boot = np.stack([auc(q[r, DEMENTIA][_bp], q[r, DEMENTIA][_bn]) for r in range(N_REPEATS)]).mean(0)
    rows.append(dict(run=n, auc=pr.auc.mean(), auc_lo=np.percentile(boot, 2.5), auc_hi=np.percentile(boot, 97.5),
                     auc_sd_repeats=pr.auc.std(ddof=1), bal_acc=pr.bal_acc.mean(),
                     recall_AD=pr.recall_AD.mean(), recall_FTD=pr.recall_FTD.mean()))
RESULTS = pd.DataFrame(rows).set_index("run")
pd.set_option("display.width", 200)
print("C3, AD vs FTD, 59 patients, 5 x 5-fold (Experiment 9's folds); mean over repeats\n")
print(RESULTS.round(3).to_string())


C3, AD vs FTD, 59 patients, 5 x 5-fold (Experiment 9's folds); mean over repeats

                                            auc  auc_lo  auc_hi  auc_sd_repeats  bal_acc  recall_AD  recall_FTD
run                                                                                                            
ds006036, 44 s (primary)                  0.641   0.515   0.753           0.036    0.597      0.656       0.539
ds004504, same 44 s (control)             0.523   0.408   0.637           0.050    0.499      0.650       0.348
ds004504, 300 s (Experiment 9, post hoc)  0.628   0.502   0.748           0.038    0.605      0.722       0.487


In [7]:
# ===========================================================================
# CELL 7 — PRIMARY TEST: THE WHOLE PIPELINE ON SHUFFLED DIAGNOSES (all 5 repeats)
# ===========================================================================
PRIMARY = "ds006036, 44 s (primary)"


def shuffled_auc(k):
    labels = LABELS.copy()
    labels[DEMENTIA] = np.random.default_rng(SEED + 10_000 + k).permutation(labels[DEMENTIA])
    q = run_cv(XCZ_06, labels)[:, DEMENTIA]
    yd = labels[DEMENTIA] == AD
    return auc(q[:, yd], q[:, ~yd]).mean()


t0 = time.time()
NULL = np.array(Parallel(n_jobs=N_JOBS)(delayed(shuffled_auc)(k) for k in range(N_PERM)))
OBS = RESULTS.loc[PRIMARY, "auc"]
P_PERM = (1 + (NULL >= OBS).sum()) / (1 + len(NULL))
CONFIRMED = bool(P_PERM < 0.05)
print(f"{len(NULL)} shuffles in {time.time() - t0:.0f} s")
print(f"ds006036 C3 AUC {OBS:.3f}; shuffled: median {np.median(NULL):.3f}, 95 % range "
      f"{np.percentile(NULL, 2.5):.3f}-{np.percentile(NULL, 97.5):.3f}; one-sided p = {P_PERM:.4f}")
print(f"\nC3 {'CONFIRMED' if CONFIRMED else 'NOT CONFIRMED'} on ds006036 (pre-registered rule: p < 0.05)")


1000 shuffles in 884 s
ds006036 C3 AUC 0.641; shuffled: median 0.488, 95 % range 0.320-0.658; one-sided p = 0.0450

C3 CONFIRMED on ds006036 (pre-registered rule: p < 0.05)


In [8]:
# ===========================================================================
# CELL 8 — SECONDARY: SEVERITY, ICA RANK AND AGREEMENT BETWEEN THE TWO RECORDINGS
# ===========================================================================
_s = subjects.iloc[DEMENTIA]


def loglik(Xc, y):
    m = LogisticRegression(C=np.inf, max_iter=10000).fit(Xc, y)          # unpenalised
    p = np.clip(m.predict_proba(Xc)[:, 1], 1e-12, 1 - 1e-12)
    return (y * np.log(p) + (1 - y) * np.log(1 - p)).sum()


_base = StandardScaler().fit_transform(np.column_stack([_s.mmse, _s.age, _s.sex == "F"]).astype(float))
_ll0 = loglik(_base, YD)
conf = []
for n, q in Q.items():
    qm = np.clip(q[:, DEMENTIA].mean(0), 1e-6, 1 - 1e-6)
    lq = np.log(qm / (1 - qm))
    lr = 2 * (loglik(np.column_stack([_base, (lq - lq.mean()) / lq.std()]), YD) - _ll0)
    conf.append(dict(run=n, lr_chi2=lr, p_beyond_mmse_age_sex=stats.chi2.sf(lr, 1),
                     r_with_mmse=stats.spearmanr(qm, _s.mmse)[0]))
CONFOUNDS = pd.DataFrame(conf).set_index("run")
print("does each score add to MMSE + age + sex? (likelihood-ratio test, 1 df)\n")
print(CONFOUNDS.round(3).to_string())

_deficit = 19.0 - RANK_06[DEMENTIA]
RANK_AUC = auc(_deficit[POS], _deficit[NEG])
_q06, _q04 = Q[PRIMARY][:, DEMENTIA].mean(0), C3_EXP9[:, DEMENTIA].mean(0)
AGREE = stats.spearmanr(_q06, _q04)
print(f"\nds006036 ICA rank deficit alone, AD vs FTD AUC {RANK_AUC:.3f}")
print(f"agreement: Spearman r between each patient's C3 score on ds006036 and on ds004504 = "
      f"{AGREE[0]:.3f} (p {AGREE[1]:.3f}, 59 patients)")


does each score add to MMSE + age + sex? (likelihood-ratio test, 1 df)

                                          lr_chi2  p_beyond_mmse_age_sex  r_with_mmse
run                                                                                  
ds006036, 44 s (primary)                    1.782                  0.182       -0.274
ds004504, same 44 s (control)               0.128                  0.720        0.027
ds004504, 300 s (Experiment 9, post hoc)    0.460                  0.498       -0.361

ds006036 ICA rank deficit alone, AD vs FTD AUC 0.554
agreement: Spearman r between each patient's C3 score on ds006036 and on ds004504 = 0.399 (p 0.002, 59 patients)


In [9]:
# ===========================================================================
# CELL 9 — SAVE
# ===========================================================================
np.savez_compressed(OUT_DIR / "c3_oof.npz", subjects=subjects.participant_id.to_numpy(), labels=LABELS,
                    runs=np.array(list(Q)), q_ad=np.stack([Q[n] for n in Q]), fold_of=FOLD_OF)
RESULTS.to_csv(OUT_DIR / "results.csv")
CONFOUNDS.to_csv(OUT_DIR / "confounds.csv")
pd.DataFrame(dict(shuffled_auc=NULL)).to_csv(OUT_DIR / "permutation.csv", index=False)
_p = RESULTS.loc[PRIMARY]
SUMMARY = dict(task="AD vs FTD, 36 vs 23: confirm C3 (picked post hoc in Experiment 9) on ds006036",
               protocol=PROTOCOL["written"], data=f"{N_EPOCHS} x {EPOCH_S:g} s per subject",
               primary=dict(auc=round(_p.auc, 4), ci=[round(_p.auc_lo, 4), round(_p.auc_hi, 4)],
                            bal_acc=round(_p.bal_acc, 4), recall_AD=round(_p.recall_AD, 4),
                            recall_FTD=round(_p.recall_FTD, 4), p_permutation=round(P_PERM, 4),
                            n_perm=N_PERM, confirmed=CONFIRMED),
               runs={n: dict(auc=round(r.auc, 4), ci=[round(r.auc_lo, 4), round(r.auc_hi, 4)],
                             p_beyond_mmse_age_sex=round(CONFOUNDS.loc[n, "p_beyond_mmse_age_sex"], 4))
                     for n, r in RESULTS.iterrows()},
               agreement_spearman=[round(AGREE[0], 4), round(AGREE[1], 4)], rank_deficit_auc=round(RANK_AUC, 4),
               comparators="Experiment 9 C3 0.628 (post hoc); Mlinaric et al. 2026 0.651")
(OUT_DIR / "summary.json").write_text(json.dumps(SUMMARY, indent=1, ensure_ascii=False, default=float),
                                      encoding="utf-8")
print(f"saved to {OUT_DIR}:", ", ".join(sorted(p.name for p in OUT_DIR.iterdir())))


saved to C:\Projects\Major Project\experiments_ad_vs_ftd\results\c3_ds006036: c3_oof.npz, confounds.csv, permutation.csv, protocol.json, results.csv, summary.json
